# Week 5 Day 3 — LightGBM

## What I built:
- LightGBM baseline on House Prices
- Compared LightGBM vs XGBoost vs Random Forest
- Tuned LightGBM with RandomizedSearchCV — 20 combinations
- Submitted to Kaggle — new best score

## LightGBM vs XGBoost:
XGBoost  → level-wise tree growth
LightGBM → leaf-wise tree growth → faster, sometimes better

## LightGBM-specific parameters:
- num_leaves       = 63   ← controls tree complexity (not max_depth)
- min_child_samples= 20   ← minimum samples per leaf
- reg_alpha        = 0.5  ← L1 regularization
- reg_lambda       = 0.1  ← L2 regularization
- learning_rate    = 0.005 ← very small → needs 2000 trees

## Results:
CV RMSE comparison:
  Random Forest  : 0.1315
  XGBoost tuned  : 0.1220
  LightGBM tuned : 0.1274

Kaggle scores:
  XGBoost        : 0.12930
  LightGBM       : 0.12603  ← new best 🏆

## Key insight:
CV score ≠ Kaggle score always
LightGBM had worse CV (0.1274 vs 0.1220) but better Kaggle score
Always submit both and compare on actual test data

## LightGBM vs XGBoost — Tree Growth Strategy:

**XGBoost — Level-wise growth:**
- Grows all nodes at the same depth level before going deeper
- More stable, less prone to overfitting on small datasets
- Slower — has to process entire level even if some splits are useless

**LightGBM — Leaf-wise growth:**
- Finds the single leaf with highest loss reduction and splits it
- Skips useless splits entirely → faster
- Can overfit on small datasets (use min_child_samples to control)

**Result:**
- LightGBM → 5-10x faster training than XGBoost
- LightGBM → sometimes better accuracy on large datasets
- LightGBM → uses less memory
- XGBoost  → more stable on small/noisy datasets

**When to use which:**
- Large dataset (>10k rows) → LightGBM first choice
- Small dataset (<5k rows)  → XGBoost safer
- Production speed matters  → LightGBM always
- Kaggle tabular            → try both, submit both

# Install and first LightGBM model:

In [2]:
pip install lightgbm

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
   --------------- ------------------------ 0.5/1.4 MB 981.3 kB/s eta 0:00:01
   ------------------------------ --------- 1.0/1.4 MB 1.5 MB/s eta 0:00:01
   -------------------------------------- - 1.3/1.4 MB 1.5 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 1.4 MB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.


In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score, KFold, train_test_split
import warnings
warnings.filterwarnings('ignore')

# load data
train = pd.read_csv('house_prices_train.csv')
test  = pd.read_csv('house_prices_test.csv')

def engineer_features(df):
    df = df.copy()
    df['TotalSF']     = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['TotalBath']   = (df['FullBath'] + 0.5*df['HalfBath'] +
                         df['BsmtFullBath'] + 0.5*df['BsmtHalfBath'])
    df['HouseAge']    = df['YrSold'] - df['YearBuilt']
    df['RemodAge']    = df['YrSold'] - df['YearRemodAdd']
    df['HasPool']     = (df['PoolArea'] > 0).astype(int)
    df['HasGarage']   = (df['GarageArea'] > 0).astype(int)
    df['HasBasement'] = (df['TotalBsmtSF'] > 0).astype(int)
    return df

train_fe = engineer_features(train)
test_fe  = engineer_features(test)

X = train_fe.drop(columns=['SalePrice', 'Id'])
y = np.log1p(train['SalePrice'])

num_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

# preprocessor
num_tr = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])
cat_tr = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value',
                               unknown_value=-1))
])
preprocessor = ColumnTransformer(transformers=[
    ('num', num_tr, num_cols),
    ('cat', cat_tr, cat_cols)
])

# LightGBM pipeline
lgb_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', lgb.LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.01,
        max_depth=5,
        num_leaves=31,
        subsample=0.7,
        colsample_bytree=0.6,
        random_state=42,
        verbose=-1
    ))
])

# evaluate
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores_lgb = cross_val_score(lgb_pipe, X, y,
                              cv=kfold,
                              scoring='neg_root_mean_squared_error')

print("=" * 55)
print("   LIGHTGBM vs XGBOOST — HOUSE PRICES")
print("=" * 55)
print(f"  LightGBM RMSE : {-scores_lgb.mean():.4f} ± {scores_lgb.std():.4f}")
print(f"  XGBoost RMSE  : 0.1220")
print(f"  RF RMSE       : 0.1315")
print("=" * 55)

   LIGHTGBM vs XGBOOST — HOUSE PRICES
  LightGBM RMSE : 0.1278 ± 0.0180
  XGBoost RMSE  : 0.1220
  RF RMSE       : 0.1315


# Tune LightGBM-specific params to push below 0.1220:

In [9]:
from sklearn.model_selection import RandomizedSearchCV

param_dist_lgb = {
    'model__num_leaves'        : [20, 31, 50, 63, 100],
    'model__learning_rate'     : [0.005, 0.01, 0.05],
    'model__n_estimators'      : [1000, 1500, 2000],
    'model__subsample'         : [0.6, 0.7, 0.8],
    'model__colsample_bytree'  : [0.5, 0.6, 0.7],
    'model__min_child_samples' : [5, 10, 20],
    'model__reg_alpha'         : [0, 0.1, 0.5],
    'model__reg_lambda'        : [0, 0.1, 1.0]
}

search_lgb = RandomizedSearchCV(
    lgb_pipe, param_dist_lgb,
    n_iter=20,
    cv=kfold,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1,
    verbose=1
)
search_lgb.fit(X, y)

print("=" * 50)
print("   TUNED LIGHTGBM")
print("=" * 50)
print(f"  Best RMSE   : {-search_lgb.best_score_:.4f}")
print(f"  Best params : {search_lgb.best_params_}")
print("=" * 50)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
   TUNED LIGHTGBM
  Best RMSE   : 0.1274
  Best params : {'model__subsample': 0.8, 'model__reg_lambda': 0.1, 'model__reg_alpha': 0.5, 'model__num_leaves': 63, 'model__n_estimators': 2000, 'model__min_child_samples': 20, 'model__learning_rate': 0.005, 'model__colsample_bytree': 0.5}


#  Submit to Kaggle:

In [11]:
# fit best LightGBM on full data
best_lgb = search_lgb.best_estimator_
best_lgb.fit(X, y)

# predict
X_test_hp  = test_fe.drop(columns=['Id'])
log_preds  = best_lgb.predict(X_test_hp)
predictions = np.expm1(log_preds)

submission = pd.DataFrame({
    'Id'       : test['Id'],
    'SalePrice': predictions
})
submission.to_csv('lgb_submission_v1.csv', index=False)
print("LightGBM submission saved!")

LightGBM submission saved!
